# Propensity to Donate: Model Evaluation & Cross-Validation
Following the textbook guidelines for small/medium datasets, we will freeze a final test set once, then use cross-validation on the training data for model selection and tuning. We will compare different cross-validation strategies (`KFold`, `StratifiedKFold`, and `RepeatedStratifiedKFold`) to see which provides the most reliable estimate for our class distribution, and then use the best one in a `GridSearchCV`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load the dataset
df = pd.read_csv('propensity_model_dataset.csv')

# 2. Separate Features (X) and Target (y)
# Drop supporter_id as it is an identifier, not a predictor
X = df.drop(columns=['supporter_id', 'target_donated_next_90_days'])
y = df['target_donated_next_90_days']

print(f"Target distribution:\n{y.value_counts(normalize=True)}")

Target distribution:
target_donated_next_90_days
0    0.610169
1    0.389831
Name: proportion, dtype: float64


## 1. Train / Test Split (Hold-out Set)
We split the data 80/20. We use `stratify=y` to ensure our test set has the same proportion of churned/retained donors as our training set.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (47, 6)
Test set size: (12, 6)


## 2. Preprocessing Pipeline
As recommended in the textbook, we process numeric and categorical variables inside a pipeline to prevent data leakage during cross-validation.

In [3]:
numeric_features = X.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create a baseline Random Forest model pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

## 3. Compare Cross-Validation Methods
We test three methods from the textbook:
- **KFold:** Standard random splits.
- **StratifiedKFold:** Preserves the percentage of samples for each class.
- **RepeatedStratifiedKFold:** Repeats Stratified K-Fold multiple times for a more stable estimate on small datasets.

In [4]:
cv_methods = {
    'KFold (5 splits)': KFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold (5 splits)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedStratifiedKFold (5 splits, 3 repeats)': RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
}

print("Cross-Validation ROC AUC Scores on Training Data:")
for name, cv in cv_methods.items():
    # We use roc_auc as our scoring metric to account for class balance
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

Cross-Validation ROC AUC Scores on Training Data:
KFold (5 splits): 0.3007 +/- 0.1235
StratifiedKFold (5 splits): 0.4289 +/- 0.1478
RepeatedStratifiedKFold (5 splits, 3 repeats): 0.4384 +/- 0.1588


## 4. Hyperparameter Tuning with the Best CV Method
Because our dataset is small, `RepeatedStratifiedKFold` generally gives the most stable evaluation. We will use it inside `GridSearchCV` to find the best hyperparameters for our Random Forest.

In [5]:
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 3, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 5]
}

best_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=best_cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validated ROC AUC: {grid_search.best_score_:.4f}")

Best Hyperparameters: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 50}
Best Cross-Validated ROC AUC: 0.4576


## 5. Final Evaluation on the Frozen Test Set
Now that we have selected our best model using cross-validation on the training set, we evaluate it **once** on the held-out test set to get an unbiased estimate of its generalization performance.

In [6]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Final Test Set Evaluation:\n")
print(classification_report(y_test, y_pred))
print(f"Test ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

Final Test Set Evaluation:

              precision    recall  f1-score   support

           0       0.43      0.43      0.43         7
           1       0.20      0.20      0.20         5

    accuracy                           0.33        12
   macro avg       0.31      0.31      0.31        12
weighted avg       0.33      0.33      0.33        12

Test ROC AUC: 0.3429


In [ ]:
import joblib

# Export the best model pipeline 
# (This saves the Random Forest AND the preprocessing steps!)
filename = "propensity_to_donate_pipeline.pkl"
joblib.dump(best_model, filename)

print(f"✅ Successfully exported the entire prediction pipeline!")
print(f"📦 Saved as: {filename}")